# 🔌 Conexión remota para agentes (Antigravity, Claude Code, Codex, Gemini CLI…)
1. **Entorno de ejecución → Cambiar tipo** → elige tu GPU grande.
2. (Opcional) ejecuta la celda 1 para usar tu Google Drive como disco persistente.
3. Ejecuta la celda 2 y **déjala corriendo**.
4. Copia la dirección `xxxx.trycloudflare.com` y pégasela al agente.

In [ ]:
#@title 1️⃣ (Opcional) Montar Google Drive — los archivos no se pierden al cerrar Colab
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/hyperframes-studio', exist_ok=True)
print('Carpeta persistente: /content/drive/MyDrive/hyperframes-studio')

In [ ]:
#@title 2️⃣ Abrir conexión SSH (dejar corriendo)
PUBKEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIBFfwbNKZCi7gWxq5WiZ7ABTs5vZw1O/JfF3SkE2j4BW antigravity-colab"

import os, re, subprocess, time, shlex
sh = lambda c: subprocess.run(c, shell=True, check=False)

# Servidor SSH solo con llave (sin contraseña)
sh("apt-get -qq update >/dev/null && apt-get -qq install -y openssh-server rsync ffmpeg >/dev/null")
os.makedirs('/root/.ssh', exist_ok=True)
open('/root/.ssh/authorized_keys', 'w').write(PUBKEY + '\n')
os.chmod('/root/.ssh', 0o700); os.chmod('/root/.ssh/authorized_keys', 0o600)
sh("mkdir -p /run/sshd && printf 'PasswordAuthentication no\nPermitRootLogin prohibit-password\nClientAliveInterval 30\n' >> /etc/ssh/sshd_config && /usr/sbin/sshd")

# Que la terminal SSH vea la GPU igual que el notebook
with open('/root/.colab_env', 'w') as f:
    for k, v in os.environ.items():
        if k.isidentifier() and k not in ('HOME', 'PWD', 'SHLVL', '_'):
            f.write(f'export {k}={shlex.quote(v)}\n')
sh("grep -q colab_env /root/.bashrc || echo 'source /root/.colab_env; cd /content' >> /root/.bashrc")

# Túnel de Cloudflare (no hace falta cuenta)
sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
p = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'ssh://127.0.0.1:2222'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
host = None
for line in p.stdout:
    m = re.search(r'https://([a-z0-9-]+\.trycloudflare\.com)', line)
    if m:
        host = m.group(1); break

print('\n' + '=' * 60)
print('✅ COPIA ESTO Y PÉGASELO AL AGENTE:\n')
print('   ', host)
print('=' * 60 + '\n')
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; echo CPUs: $(nproc); free -g | head -2")

# Drenar los logs sin bloquear las siguientes celdas.
import threading
def guardar_logs_tunel():
    with open('/content/cloudflared.log', 'a', buffering=1) as registro:
        for linea in p.stdout:
            registro.write(linea)
threading.Thread(target=guardar_logs_tunel, daemon=True).start()
print('Tunel en segundo plano. Ya puedes ejecutar otras celdas.')
